#### Notebook 3 - Merge and Clean Dataset

<small>
Notebook ini bertujuan untuk menggabungkan seluruh dataset yang telah diproses menjadi satu dataset utama.

Tahapan yang dilakukan meliputi:

- Load processed dataset
- Merge dataset
- Membersihkan teks
- Menghapus duplicate
- Menyimpan dataset final</small>

In [100]:
import pandas as pd
import matplotlib.pyplot as plt
import re

In [101]:
df1 = pd.read_csv("../data/processed/train_clean.csv")
df2 = pd.read_csv("../data/processed/test_clean.csv")
df3 = pd.read_csv("../data/processed/finance_clean.csv")
df4 = pd.read_csv("../data/processed/transaction_clean.csv")

df5 = pd.read_csv("../data/processed/transport_clean.csv")
df6 = pd.read_csv("../data/processed/transfer_clean.csv")
df7 = pd.read_csv("../data/processed/topup_clean.csv")
df8 = pd.read_csv("../data/processed/donation_clean.csv")
df9 = pd.read_csv("../data/processed/fees_clean.csv")

df10 = pd.read_csv("../data/processed/manual_clean.csv")

In [102]:
merged = pd.concat(
    [
        df1,
        df2,
        df3,
        df4,
        df5,
        df6,
        df7,
        df8,
        df9,
        df10
    ],
    ignore_index=True
)

In [103]:
import re

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

merged["clean_text"] = merged["text"].apply(clean_text)

In [104]:
print(merged.columns)

Index(['text', 'label', 'clean_text'], dtype='object')


In [105]:
merged.head()

,text,label,clean_text
0,Pembayaran restoran sebesar Rp 48698 via UPI R...,food,pembayaran restoran sebesar rp 48698 via upi r...
1,Pembelian tiket kereta sebesar Rp 9244 via UPI...,travel,pembelian tiket kereta sebesar rp 9244 via upi...
2,Pembelian paket makan siang sebesar Rp 5797 vi...,food,pembelian paket makan siang sebesar rp 5797 vi...
3,SIP reksa dana sebesar Rp 2052 via UPI Ref 198246,investment,sip reksa dana sebesar rp 2052 via upi ref 198246
4,Pembelian tiket kereta sebesar Rp 33218 via UP...,travel,pembelian tiket kereta sebesar rp 33218 via up...


In [106]:
merged = merged.drop_duplicates(
    subset=[
        "clean_text",
        "label"
    ]
)

In [107]:
print(merged.shape)

(3906, 3)


In [108]:
merged.head()

,text,label,clean_text
0,Pembayaran restoran sebesar Rp 48698 via UPI R...,food,pembayaran restoran sebesar rp 48698 via upi r...
1,Pembelian tiket kereta sebesar Rp 9244 via UPI...,travel,pembelian tiket kereta sebesar rp 9244 via upi...
2,Pembelian paket makan siang sebesar Rp 5797 vi...,food,pembelian paket makan siang sebesar rp 5797 vi...
3,SIP reksa dana sebesar Rp 2052 via UPI Ref 198246,investment,sip reksa dana sebesar rp 2052 via upi ref 198246
4,Pembelian tiket kereta sebesar Rp 33218 via UP...,travel,pembelian tiket kereta sebesar rp 33218 via up...


In [109]:
merged["label"].value_counts()

label
education        515
healthcare       451
transfer         353
shopping         321
bills            321
entertainment    246
food             224
topup            224
transport        219
donation         219
income           196
loan             183
fees             159
travel           147
investment       128
Name: count, dtype: int64

##### Text Cleaning

Membersihkan teks agar format seluruh transaksi menjadi konsisten sebelum dilakukan ekstraksi fitur menggunakan TF-IDF.

In [110]:
def clean_text(text):

    text = str(text).lower()

    text = re.sub(r"http\S+", "", text)

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

In [111]:
merged["clean_text"] = merged["text"].apply(clean_text)

In [112]:
merged[
    ["text","clean_text"]
].head()

,text,clean_text
0,Pembayaran restoran sebesar Rp 48698 via UPI R...,pembayaran restoran sebesar rp 48698 via upi r...
1,Pembelian tiket kereta sebesar Rp 9244 via UPI...,pembelian tiket kereta sebesar rp 9244 via upi...
2,Pembelian paket makan siang sebesar Rp 5797 vi...,pembelian paket makan siang sebesar rp 5797 vi...
3,SIP reksa dana sebesar Rp 2052 via UPI Ref 198246,sip reksa dana sebesar rp 2052 via upi ref 198246
4,Pembelian tiket kereta sebesar Rp 33218 via UP...,pembelian tiket kereta sebesar rp 33218 via up...


In [113]:
merged.isnull().sum()

text          0
label         0
clean_text    0
dtype: int64

In [114]:
merged = merged.reset_index(drop=True)

In [115]:
print(merged.shape)

(3906, 3)


In [116]:
print(merged["label"].value_counts())

label
education        515
healthcare       451
transfer         353
shopping         321
bills            321
entertainment    246
food             224
topup            224
transport        219
donation         219
income           196
loan             183
fees             159
travel           147
investment       128
Name: count, dtype: int64


In [117]:
print(sorted(merged["label"].unique()))

['bills', 'donation', 'education', 'entertainment', 'fees', 'food', 'healthcare', 'income', 'investment', 'loan', 'shopping', 'topup', 'transfer', 'transport', 'travel']


In [118]:
for label in sorted(merged["label"].unique()):

    print("="*60)

    print(label)

    print(
        merged[
            merged["label"]==label
        ]["text"].sample(
            min(5,len(merged[merged["label"]==label])),
            random_state=42
        ).tolist()
    )

bills
['Bayar tagihan e-wallet', 'Bayar tagihan smart home', 'Bayar tagihan publication', 'Arisan', 'Bayar tagihan air']
donation
['Bayar infak kebaikan', 'Donasi untuk pengungsi', 'Infak di bulan Ramadan', 'Sumbangan pendidikan', 'Donasi bencana gempa']
education
['Bayar kursus industrial design', 'Sertifikasi SAP', 'TOEFL prediction test', 'Beli buku public relations', 'Sertifikasi Microsoft']
entertainment
['Steam Wallet Rp200000', 'Netflix', 'Langganan Sundance Now', 'Top up game Valorant points', 'Top up game Perfect World Mobile']
fees
['Biaya admin pembayaran internet', 'Potongan admin maintenance fee', 'Potongan admin annual fee', 'Potongan admin tebus', 'Biaya admin rekening']
food
['Pembayaran restoran sebesar Rp 7285 via UPI Ref 260265', 'Pembayaran restoran sebesar Rp 19774 via UPI Ref 797118', 'Tagihan kafe sebesar Rp 35424 via UPI Ref 679104', 'Pembelian di toko kelontong sebesar Rp 39663 via UPI Ref 174357', 'Bakso Pak Kumis jam 7 malam']
healthcare
['Beli obat difteri',

In [119]:
merged.to_csv(
    "../data/final/merged_dataset.csv",
    index=False
)

In [120]:
merged.head()

,text,label,clean_text
0,Pembayaran restoran sebesar Rp 48698 via UPI R...,food,pembayaran restoran sebesar rp 48698 via upi r...
1,Pembelian tiket kereta sebesar Rp 9244 via UPI...,travel,pembelian tiket kereta sebesar rp 9244 via upi...
2,Pembelian paket makan siang sebesar Rp 5797 vi...,food,pembelian paket makan siang sebesar rp 5797 vi...
3,SIP reksa dana sebesar Rp 2052 via UPI Ref 198246,investment,sip reksa dana sebesar rp 2052 via upi ref 198246
4,Pembelian tiket kereta sebesar Rp 33218 via UP...,travel,pembelian tiket kereta sebesar rp 33218 via up...


In [121]:
merged[merged["label"]=="food"].sample(30, random_state=42)["text"].tolist()

['Pembayaran restoran sebesar Rp 7285 via UPI Ref 260265',
 'Pembayaran restoran sebesar Rp 19774 via UPI Ref 797118',
 'Tagihan kafe sebesar Rp 35424 via UPI Ref 679104',
 'Pembelian di toko kelontong sebesar Rp 39663 via UPI Ref 174357',
 'Bakso Pak Kumis jam 7 malam',
 'Tagihan kafe sebesar Rp 26695 via UPI Ref 409890',
 'Pembelian di toko kelontong sebesar Rp 32384 via UPI Ref 221761',
 'Pembelian di toko kelontong sebesar Rp 47343 via UPI Ref 901282',
 'Bakmi GM',
 "McDonald's",
 'Tagihan kafe sebesar Rp 34076 via UPI Ref 479353',
 'Tagihan kafe sebesar Rp 14825 via UPI Ref 167136',
 'Bakso beranak isi telur',
 'Pesanan Swiggy sebesar Rp 45386 via UPI Ref 256294',
 'Pesanan Swiggy sebesar Rp 17218 via UPI Ref 287241',
 'Pembayaran restoran sebesar Rp 22854 via UPI Ref 658623',
 'Makan bakso',
 'Pembayaran toko roti sebesar Rp 30474 via UPI Ref 189365',
 'Bakso beranak kesukaan',
 'Transaksi di restoran pizza sebesar Rp 30481 via UPI Ref 266309',
 'Tagihan kafe sebesar Rp 8780 via 

In [122]:
merged[merged["label"]=="food"].sample(100, random_state=42)["text"].tolist()

['Pembayaran restoran sebesar Rp 7285 via UPI Ref 260265',
 'Pembayaran restoran sebesar Rp 19774 via UPI Ref 797118',
 'Tagihan kafe sebesar Rp 35424 via UPI Ref 679104',
 'Pembelian di toko kelontong sebesar Rp 39663 via UPI Ref 174357',
 'Bakso Pak Kumis jam 7 malam',
 'Tagihan kafe sebesar Rp 26695 via UPI Ref 409890',
 'Pembelian di toko kelontong sebesar Rp 32384 via UPI Ref 221761',
 'Pembelian di toko kelontong sebesar Rp 47343 via UPI Ref 901282',
 'Bakmi GM',
 "McDonald's",
 'Tagihan kafe sebesar Rp 34076 via UPI Ref 479353',
 'Tagihan kafe sebesar Rp 14825 via UPI Ref 167136',
 'Bakso beranak isi telur',
 'Pesanan Swiggy sebesar Rp 45386 via UPI Ref 256294',
 'Pesanan Swiggy sebesar Rp 17218 via UPI Ref 287241',
 'Pembayaran restoran sebesar Rp 22854 via UPI Ref 658623',
 'Makan bakso',
 'Pembayaran toko roti sebesar Rp 30474 via UPI Ref 189365',
 'Bakso beranak kesukaan',
 'Transaksi di restoran pizza sebesar Rp 30481 via UPI Ref 266309',
 'Tagihan kafe sebesar Rp 8780 via 

In [123]:
merged["label"].value_counts()

label
education        515
healthcare       451
transfer         353
shopping         321
bills            321
entertainment    246
food             224
topup            224
transport        219
donation         219
income           196
loan             183
fees             159
travel           147
investment       128
Name: count, dtype: int64

#### Kesimpulan
<small>
Seluruh dataset berhasil digabungkan menjadi satu dataset utama.

Dataset akhir telah melalui proses:

- Penggabungan dataset
- Pembersihan teks
- Penghapusan data duplikat
- Pemeriksaan missing value

Dataset yang dihasilkan akan digunakan pada tahap ekstraksi fitur dan pelatihan model Machine Learning.</small>